In [1]:
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
import os

In [3]:
TRAIN_DIR = "D:/college/grad project/integrated/Training"
IMG_SIZE = (224, 224)
BATCH_SIZE = 16
EPOCHS = 10
MODEL_PATH = "saved_model/mri_classifier.keras"
FINE_TUNED_PATH = "saved_model/mri_classifier_finetuned.keras"

In [5]:
model = load_model(MODEL_PATH)

# Freeze all layers except the last 5
for layer in model.layers[:-5]:
    layer.trainable = False

In [7]:
datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    zoom_range=0.1,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    validation_split=0.2  # 20% will be used for validation
)

train_generator = datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training'
)

val_generator = datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation'
)

Found 9345 images belonging to 4 classes.
Found 2333 images belonging to 4 classes.


In [9]:
model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator,
    callbacks=[early_stop]
)

D:\anaconda\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
585/585 ━━━━━━━━━━━━━━━━━━━━ 303s 508ms/step - accuracy: 0.8422 - loss: 0.7087 - val_accuracy: 0.8988 - val_loss: 0.3543
Epoch 2/10
585/585 ━━━━━━━━━━━━━━━━━━━━ 237s 406ms/step - accuracy: 0.9695 - loss: 0.0878 - val_accuracy: 0.9126 - val_loss: 0.3072
Epoch 3/10
585/585 ━━━━━━━━━━━━━━━━━━━━ 235s 401ms/step - accuracy: 0.9766 - loss: 0.0683 - val_accuracy: 0.9224 - val_loss: 0.2962
Epoch 4/10
585/585 ━━━━━━━━━━━━━━━━━━━━ 235s 402ms/step - accuracy: 0.9788 - loss: 0.0599 - val_accuracy: 0.9194 - val_loss: 0.3116
Epoch 5/10
585/585 ━━━━━━━━━━━━━━━━━━━━ 235s 402ms/step - accuracy: 0.9814 - loss: 0.0522 - val_accuracy: 0.9224 - val_loss: 0.3081
Epoch 6/10
585/585 ━━━━━━━━━━━━━━━━━━━━ 236s 403ms/step - accuracy: 0.9808 - loss: 0.0515 - val_accuracy: 0.9237 - val_loss: 0.3087


In [11]:
model.save(FINE_TUNED_PATH)
print(f"Fine-tuned model saved at: {FINE_TUNED_PATH}")

Fine-tuned model saved at: saved_model/mri_classifier_finetuned.keras


In [13]:
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

In [17]:
# CONFIGURATION
MODEL_PATH = "saved_model/mri_classifier_finetuned.keras"
TEST_DIR = "D:/college/grad project/integrated/testing"  # Update as needed
IMG_SIZE = (224, 224)  # Must match the input size your model was trained on
BATCH_SIZE = 32
CLASS_NAMES = ["glioma_tumor", "meningioma_tumor", "no_tumor", "pituitary_tumor"]

In [19]:
# LOAD MODEL
model = load_model(MODEL_PATH)

In [21]:
# LOAD TEST DATA
test_datagen = ImageDataGenerator(rescale=1./255)

test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False  # To keep order of labels aligned with predictions
)

Found 1705 images belonging to 4 classes.


In [23]:
# PREDICT
# -------------------------------
y_pred_probs = model.predict(test_generator)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = test_generator.classes  # Already encoded from folder names

54/54 ━━━━━━━━━━━━━━━━━━━━ 30s 524ms/step


In [25]:
# METRICS
# -------------------------------
print("Classification Report:\n")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

cm = confusion_matrix(y_true, y_pred)
per_class_accuracy = cm.diagonal() / cm.sum(axis=1)

print("\nPer-Class Accuracy:")
for i, class_name in enumerate(CLASS_NAMES):
    print(f"{class_name}: {per_class_accuracy[i]*100:.2f}%")

print("\nConfusion Matrix:")
print(cm)

Classification Report:

                  precision    recall  f1-score   support

    glioma_tumor       0.97      0.85      0.91       400
meningioma_tumor       0.92      0.94      0.93       421
        no_tumor       0.91      1.00      0.95       510
 pituitary_tumor       0.97      0.95      0.96       374

        accuracy                           0.94      1705
       macro avg       0.94      0.93      0.94      1705
    weighted avg       0.94      0.94      0.94      1705


Per-Class Accuracy:
glioma_tumor: 85.00%
meningioma_tumor: 93.59%
no_tumor: 100.00%
pituitary_tumor: 94.92%

Confusion Matrix:
[[340  25  34   1]
 [ 10 394   6  11]
 [  0   0 510   0]
 [  1   8  10 355]]


In [27]:
model.save("saved_model/mri_classifier_final.keras")